**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# E11 Parcellation — ROI Analysis

Loads the parcellation registered to GESFIDE space and extracts
per-region parameter means from the smoothed Triple DL and DM maps.

**Required files:**
- `./parcellation/e11_parc_GESFIDE.nii.gz` — parcellation in GESFIDE space
- `./parcellation/e11_GESFIDE_ref.nii.gz`  — first echo reference
- `./invivo_smooth_v1/img_e11_AIR_maps_smooth.mat` — smoothed parameter maps

## 1. Imports

In [ ]:
import os
import numpy as np
import scipy.io as sio
import nibabel as nib
import h5py
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 10,
    'axes.titlesize' : 11,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

C_DL = '#1565C0'
C_DM = '#E65100'

os.makedirs('./results/parcellation/figures', exist_ok=True)
print('Imports OK')

## 2. Paths

In [ ]:
import glob
for f in sorted(glob.glob('./results/invivo_smooth_v1/*.mat')):
    print(f)

In [ ]:
# PARC_PATH  = './results/parcellation/e11_parc_GESFIDE.nii.gz'
# PARC_PATH  = './results/parcellation/e11_parc_GESFIDE_v2.nii.gz'
# PARC_PATH = './results/parcellation/e11_parc_GESFIDE_bbr.nii.gz'
PARC_PATH = './results/parcellation/e11_parc_sform.nii.gz'
REF_PATH   = './results/parcellation/e11_GESFIDE_ref.nii.gz'
MAPS_PATH  = './results/invivo_smooth_v1/img_e11air_maps_smooth.mat'
FIG_DIR    = './results/parcellation/figures'

# Parameter spec
PARAM_NAMES = ['SO2', 'CBV', 'R',   'T2']
PARAM_SCALE = [100,   100,   1e6,   1000]
PARAM_UNITS = ['%',   '%',   'µm',  'ms']
PARAM_LABEL = ['SO₂ (%)', 'CBV (%)', 'R (µm)', 'T2 (ms)']

# GM parcellation labels (FreeSurfer aparc+aseg)
PARC_LABELS = {
    'Frontal GM'  : [1028, 1003, 1027, 2028, 2003, 2027],
    'Parietal GM' : [1008, 1025, 1029, 2008, 2025, 2029],
    'Occipital GM': [1011, 1013, 1005, 2011, 2013, 2005],
    'Temporal GM' : [1015, 1030, 1001, 2015, 2030, 2001],
    'Cingulate'   : [1002, 1010, 1023, 1026, 2002, 2010, 2023, 2026],
    'Insula'      : [1035, 2035],
    # Subcortical — non-heme iron present, interpret cautiously
    'Thalamus'    : [10,  49],
    'Caudate'     : [11,  50],
    'Putamen'     : [12,  51],
}

print('Paths set.')

## 3. Load data

In [ ]:
def load_mat_safe(path, key):
    try:
        mat = sio.loadmat(path)
        if key in mat: return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        print(f'  Key "{key}" not found, available: {cands}')
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2: data = data.T
            return np.array(data, dtype=np.float32)

# ── Parcellation ──────────────────────────────────────────────────────────
parc_img  = nib.load(PARC_PATH)
parc      = parc_img.get_fdata().astype(int)
ref       = nib.load(REF_PATH).get_fdata()

print(f'Parcellation shape : {parc.shape}')
print(f'Unique labels      : {len(np.unique(parc))}')
print(f'Non-zero voxels    : {(parc > 0).sum():,}')

# ── Smoothed maps ─────────────────────────────────────────────────────────
maps     = sio.loadmat(MAPS_PATH)
print(f'\nMap keys: {[k for k in maps if not k.startswith("_")]}')

dl_maps  = np.array(maps['Triple'])   # (H, W, S, 4)
dm_maps  = np.array(maps['DM'])
mask3d   = np.array(maps['mask']).astype(bool)

print(f'DL maps shape : {dl_maps.shape}')
print(f'DM maps shape : {dm_maps.shape}')
print(f'Mask shape    : {mask3d.shape}')
print(f'Mask voxels   : {mask3d.sum():,}')

# ── Check overlap ─────────────────────────────────────────────────────────
overlap = mask3d & (parc > 0)
print(f'\nOverlap (mask & parcellation): {overlap.sum():,} voxels')

## 4. Visualise — registration check (all 14 slices)

In [ ]:
n_sl = parc.shape[2]
fig, axes = plt.subplots(2, n_sl, figsize=(n_sl * 1.6, 4))

for sl in range(n_sl):
    # Row 0: GESFIDE reference
    axes[0, sl].imshow(ref[:, :, sl].T, cmap='gray', origin='lower')
    axes[0, sl].set_title(f'sl {sl}', fontsize=7)
    axes[0, sl].axis('off')

    # Row 1: parcellation overlay on reference
    p = parc[:, :, sl].T.astype(float)
    p[p == 0] = np.nan
    axes[1, sl].imshow(ref[:, :, sl].T, cmap='gray',  origin='lower')
    axes[1, sl].imshow(p,               cmap='tab20', origin='lower',
                       alpha=0.65, interpolation='none')
    axes[1, sl].axis('off')

axes[0, 0].set_ylabel('GESFIDE ref', fontsize=8)
axes[1, 0].set_ylabel('Parcellation', fontsize=8)
fig.suptitle('E11 — Registration check', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'e11_reg_check.png'), dpi=200, bbox_inches='tight')
plt.show()

## 5. Visualise — parameter maps + parcellation overlay

In [ ]:
CMAPS  = ['RdYlBu_r', 'turbo', 'pink', 'bone']
CLIMS  = [(50,100), (0,8), (5,25), (40,120)]
sl_mid = parc.shape[2] // 2   # middle slice — change if needed

fig, axes = plt.subplots(3, 4, figsize=(14, 9),
                         gridspec_kw={'hspace': 0.35, 'wspace': 0.32})

for ci, (pname, sc, plabel, cmap, (vlo, vhi)) in enumerate(
        zip(PARAM_NAMES, PARAM_SCALE, PARAM_LABEL, CMAPS, CLIMS)):

    dl_sl = dl_maps[:, :, sl_mid, ci] * sc
    dm_sl = dm_maps[:, :, sl_mid, ci] * sc
    p_sl  = parc[:, :, sl_mid].T.astype(float)
    p_sl[p_sl == 0] = np.nan

    # Row 0: DL map
    im = axes[0, ci].imshow(dl_sl.T, cmap=cmap, vmin=vlo, vmax=vhi, origin='lower')
    plt.colorbar(im, ax=axes[0, ci], fraction=0.046, pad=0.04)
    axes[0, ci].set_title(plabel, fontsize=10, fontweight='bold')
    axes[0, ci].axis('off')

    # Row 1: DM map
    im = axes[1, ci].imshow(dm_sl.T, cmap=cmap, vmin=vlo, vmax=vhi, origin='lower')
    plt.colorbar(im, ax=axes[1, ci], fraction=0.046, pad=0.04)
    axes[1, ci].axis('off')

    # Row 2: DL map + parcellation overlay
    axes[2, ci].imshow(dl_sl.T, cmap=cmap, vmin=vlo, vmax=vhi, origin='lower')
    axes[2, ci].imshow(p_sl,    cmap='tab20', origin='lower',
                       alpha=0.45, interpolation='none')
    axes[2, ci].axis('off')

axes[0, 0].set_ylabel('Triple DL', fontsize=9, fontweight='bold', color=C_DL)
axes[1, 0].set_ylabel('DM',        fontsize=9, fontweight='bold', color=C_DM)
axes[2, 0].set_ylabel('DL + Parc', fontsize=9, fontweight='bold')

fig.suptitle(f'E11 — Air condition  (slice {sl_mid})',
             fontsize=11, fontweight='bold')
plt.savefig(os.path.join(FIG_DIR, 'e11_maps_parcellation.png'),
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Parcellation across all slices with T1 background (black bg, coloured regions) ─
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

# Load
ref  = nib.load('./results/parcellation/e11_T1_resampled.nii.gz').get_fdata()
parc = nib.load('./results/parcellation/e11_parc_sform.nii.gz').get_fdata().astype(int)

# Build a discrete colormap — one distinct colour per unique label
unique_labels = [int(l) for l in np.unique(parc) if l != 0]
n_labels      = len(unique_labels)

# tab20 + tab20b + tab20c gives 60 distinct colours
base_cmaps   = ['tab20', 'tab20b', 'tab20c']
colors_pool  = np.vstack([plt.get_cmap(c).colors for c in base_cmaps])
rng          = np.random.default_rng(7)
rng.shuffle(colors_pool)
label_colors = colors_pool[:n_labels]
label_to_idx = {lab: i for i, lab in enumerate(unique_labels)}

# Build an RGBA overlay where 0 = transparent, otherwise coloured
parc_rgba = np.zeros(parc.shape + (4,), dtype=np.float32)
for lab, idx in label_to_idx.items():
    m = (parc == lab)
    parc_rgba[m, :3] = label_colors[idx]
    parc_rgba[m,  3] = 0.75   # opacity

n_sl = parc.shape[2]
fig, axes = plt.subplots(1, n_sl, figsize=(n_sl * 1.6, 2.4),
                         facecolor='black')

for sl in range(n_sl):
    ax = axes[sl]
    ax.set_facecolor('black')
    ax.imshow(ref[:, :, sl].T, cmap='gray', origin='lower')
    ax.imshow(parc_rgba[:, :, sl].transpose(1, 0, 2),
              origin='lower', interpolation='none')
    ax.set_title(f'sl {sl}', fontsize=8, color='white')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)

fig.suptitle('E11 — Parcellation across slices (T1 background)',
             fontsize=11, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('./results/parcellation/figures/e11_parc_all_slices.png',
            dpi=300, bbox_inches='tight', facecolor='black')
plt.show()

In [ ]:
# ── Parcellation across all slices with T1 background (L/R same colour) ──
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Load
ref  = nib.load('./results/parcellation/e11_T1_resampled.nii.gz').get_fdata()
parc = nib.load('./results/parcellation/e11_parc_sform.nii.gz').get_fdata().astype(int)

# ── FreeSurfer aparc+aseg label names (L/R map to same name) ─────────────
# FS_NAMES = {
#     # Subcortical
#     10:'Thalamus', 49:'Thalamus',
#     11:'Caudate',  50:'Caudate',
#     12:'Putamen',  51:'Putamen',
#     13:'Pallidum', 52:'Pallidum',
#     17:'Hippocampus', 53:'Hippocampus',
#     18:'Amygdala',    54:'Amygdala',
#     26:'Accumbens',   58:'Accumbens',
#     # Cortical — strip 1xxx/2xxx prefix
#     1002:'CaudAntCing',  2002:'CaudAntCing',
#     1003:'CaudMidFront', 2003:'CaudMidFront',
#     1005:'Cuneus',       2005:'Cuneus',
#     1006:'Entorhinal',   2006:'Entorhinal',
#     1007:'Fusiform',     2007:'Fusiform',
#     1008:'InfParietal',  2008:'InfParietal',
#     1009:'InfTemporal',  2009:'InfTemporal',
#     1010:'IsthCing',     2010:'IsthCing',
#     1011:'LatOccip',     2011:'LatOccip',
#     1012:'LatOrbFront',  2012:'LatOrbFront',
#     1013:'Lingual',      2013:'Lingual',
#     1014:'MedOrbFront',  2014:'MedOrbFront',
#     1015:'MidTemporal',  2015:'MidTemporal',
#     1016:'ParaHippo',    2016:'ParaHippo',
#     1017:'Paracentral',  2017:'Paracentral',
#     1018:'ParsOperc',    2018:'ParsOperc',
#     1019:'ParsOrbital',  2019:'ParsOrbital',
#     1020:'ParsTriang',   2020:'ParsTriang',
#     1021:'Pericalc',     2021:'Pericalc',
#     1022:'Postcentral',  2022:'Postcentral',
#     1023:'PostCing',     2023:'PostCing',
#     1024:'Precentral',   2024:'Precentral',
#     1025:'Precuneus',    2025:'Precuneus',
#     1026:'RostAntCing',  2026:'RostAntCing',
#     1027:'RostMidFront', 2027:'RostMidFront',
#     1028:'SupFront',     2028:'SupFront',
#     1029:'SupParietal',  2029:'SupParietal',
#     1030:'SupTemporal',  2030:'SupTemporal',
#     1031:'Supramarg',    2031:'Supramarg',
#     1034:'TransvTemp',   2034:'TransvTemp',
#     1035:'Insula',       2035:'Insula',
# }

# # ── Group L/R labels by region name → one colour per region ──────────────
# # unique_labels  = [int(l) for l in np.unique(parc) if l != 0]
# # Only keep labels that are defined in FS_NAMES (i.e. GM regions of interest)
# unique_labels = [int(l) for l in np.unique(parc) if int(l) in FS_NAMES]

# ── Use the same 7-region grouping as the analysis ────────────────────────
PARC_LABELS = {
    'Frontal GM'  : [1028, 1003, 1027, 2028, 2003, 2027],
    'Parietal GM' : [1008, 1025, 1029, 2008, 2025, 2029],
    'Occipital GM': [1011, 1013, 1005, 2011, 2013, 2005],
    'Temporal GM' : [1015, 1030, 1001, 2015, 2030, 2001],
    'Cingulate'   : [1002, 1010, 1023, 1026, 2002, 2010, 2023, 2026],
    'Insula'      : [1035, 2035],
    'Caudate'     : [11, 50],
}

# Reverse map: individual label → grouped region name
FS_NAMES = {lab: region for region, labels in PARC_LABELS.items()
            for lab in labels}

# Only keep labels that belong to one of the 7 grouped regions
unique_labels = [int(l) for l in np.unique(parc) if int(l) in FS_NAMES]

region_names   = [FS_NAMES.get(l, f'L{l}') for l in unique_labels]
unique_regions = sorted(set(region_names), key=region_names.index)

base_cmaps   = ['tab20', 'tab20b', 'tab20c']
colors_pool  = np.vstack([plt.get_cmap(c).colors for c in base_cmaps])
rng          = np.random.default_rng(7)
rng.shuffle(colors_pool)
region_color = {name: colors_pool[i % len(colors_pool)]
                for i, name in enumerate(unique_regions)}

# Map each label → its region's colour (L and R share)
label_color  = {lab: region_color[FS_NAMES.get(lab, f'L{lab}')]
                for lab in unique_labels}

# ── Build RGBA overlay ────────────────────────────────────────────────────
parc_rgba = np.zeros(parc.shape + (4,), dtype=np.float32)
for lab, col in label_color.items():
    m = (parc == lab)
    parc_rgba[m, :3] = col
    parc_rgba[m,  3] = 0.75

# ── Plot ──────────────────────────────────────────────────────────────────
n_sl = parc.shape[2]
fig, axes = plt.subplots(1, n_sl, figsize=(n_sl * 1.6, 2.4),
                         facecolor='black')

for sl in range(n_sl):
    ax = axes[sl]
    ax.set_facecolor('black')
    ax.imshow(ref[:, :, sl].T, cmap='gray', origin='lower')
    ax.imshow(parc_rgba[:, :, sl].transpose(1, 0, 2),
              origin='lower', interpolation='none')
    ax.set_title(f'sl {sl}', fontsize=8, color='white')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)

# ── Legend: one entry per region (L/R combined) ──────────────────────────
handles = [Patch(facecolor=col, edgecolor='none', label=name)
           for name, col in region_color.items()]
fig.legend(handles=handles, loc='lower center',
           bbox_to_anchor=(0.5, -0.25),
           ncol=6, fontsize=6.5, frameon=False, labelcolor='white')

fig.suptitle('E11 — Parcellation across slices (L/R unified colours)',
             fontsize=11, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('./results/parcellation/figures/e11_parc_all_slices.png',
            dpi=300, bbox_inches='tight', facecolor='black')
plt.show()

In [ ]:
# ── Parcellation across all slices — 7 GM regions, bold colours ──────────
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# ── Load ──────────────────────────────────────────────────────────────────
ref  = nib.load('./results/parcellation/e11_T1_resampled.nii.gz').get_fdata()
parc = nib.load('./results/parcellation/e11_parc_sform.nii.gz').get_fdata().astype(int)

# ── 7-region grouping (matches analysis table) ───────────────────────────
PARC_LABELS = {
    'Frontal GM'  : [1028, 1003, 1027, 2028, 2003, 2027],
    'Parietal GM' : [1008, 1025, 1029, 2008, 2025, 2029],
    'Occipital GM': [1011, 1013, 1005, 2011, 2013, 2005],
    'Temporal GM' : [1015, 1030, 1001, 2015, 2030, 2001],
    'Cingulate'   : [1002, 1010, 1023, 1026, 2002, 2010, 2023, 2026],
    'Insula'      : [1035, 2035],
    'Caudate'     : [11, 50],
}

# Reverse map: individual label → grouped region name
FS_NAMES = {lab: region for region, labels in PARC_LABELS.items()
            for lab in labels}

# Keep only labels present AND in our 7-region grouping
unique_labels  = [int(l) for l in np.unique(parc) if int(l) in FS_NAMES]
region_names   = [FS_NAMES[l] for l in unique_labels]
unique_regions = sorted(set(region_names), key=region_names.index)

# ── High-contrast saturated palette ──────────────────────────────────────
BOLD_COLORS = [
    '#FF1744',   # bold red       — Frontal GM
    '#00E5FF',   # cyan           — Parietal GM
    '#FFEA00',   # yellow         — Occipital GM
    '#76FF03',   # bright green   — Temporal GM
    '#D500F9',   # magenta        — Cingulate
    '#FF9100',   # orange         — Insula
    '#3D5AFE',   # bold blue      — Caudate
]
colors_pool  = np.array([mcolors.to_rgb(c) for c in BOLD_COLORS])
region_color = {name: colors_pool[i % len(colors_pool)]
                for i, name in enumerate(unique_regions)}
label_color  = {lab: region_color[FS_NAMES[lab]] for lab in unique_labels}

# ── Build RGBA overlay (bold opacity) ────────────────────────────────────
parc_rgba = np.zeros(parc.shape + (4,), dtype=np.float32)
for lab, col in label_color.items():
    m = (parc == lab)
    parc_rgba[m, :3] = col
    parc_rgba[m,  3] = 0.85

# ── Plot ──────────────────────────────────────────────────────────────────
n_sl = parc.shape[2]
fig, axes = plt.subplots(1, n_sl, figsize=(n_sl * 1.6, 2.4),
                         facecolor='black')

for sl in range(n_sl):
    ax = axes[sl]
    ax.set_facecolor('black')
    ax.imshow(ref[:, :, sl].T,  cmap='gray',  origin='lower', alpha=0.55)
    ax.imshow(parc_rgba[:, :, sl].transpose(1, 0, 2),
              origin='lower', interpolation='none')
    ax.set_title(f'sl {sl}', fontsize=8, color='white')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)

# ── Legend ────────────────────────────────────────────────────────────────
handles = [Patch(facecolor=col, edgecolor='white', linewidth=0.4, label=name)
           for name, col in region_color.items()]
fig.legend(handles=handles, loc='lower center',
           bbox_to_anchor=(0.5, -0.18),
           ncol=7, fontsize=9, frameon=False, labelcolor='white')

fig.suptitle('E11 — Parcellation across slices (7 GM regions)',
             fontsize=11, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('./results/parcellation/figures/e11_parc_all_slices_bold.png',
            dpi=300, bbox_inches='tight', facecolor='black')
plt.show()

## 6. Extract ROI means per parcellation region

In [ ]:
rows = []
for region, labels in PARC_LABELS.items():
    roi_mask = np.isin(parc, labels) & mask3d
    n_vox = roi_mask.sum()
    if n_vox < 20:
        print(f'{region:<15}: only {n_vox} voxels in FOV — skipping')
        continue
    row = {'Region': region, 'N_voxels': n_vox}
    for pi, (pname, sc, punit) in enumerate(zip(PARAM_NAMES, PARAM_SCALE, PARAM_UNITS)):
        dl_vals = dl_maps[..., pi][roi_mask] * sc
        dm_vals = dm_maps[..., pi][roi_mask] * sc
        dl_vals = dl_vals[np.isfinite(dl_vals)]
        dm_vals = dm_vals[np.isfinite(dm_vals)]
        row[f'DL_{pname}_mean'] = np.nanmean(dl_vals)
        row[f'DL_{pname}_std']  = np.nanstd(dl_vals)
        row[f'DM_{pname}_mean'] = np.nanmean(dm_vals)
        row[f'DM_{pname}_std']  = np.nanstd(dm_vals)
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(os.path.join(FIG_DIR, 'e11_roi_means.csv'), index=False)

# ── Print summary ─────────────────────────────────────────────────────────
print(f'{"Region":<15} {"vox":>5}  ' +
      '  '.join(f'DL-{p:<5} DM-{p:<5}' for p in PARAM_NAMES))
print('-' * 85)
for _, r in df.iterrows():
    row_str = f'{r["Region"]:<15} {r["N_voxels"]:>5}  '
    for pname in PARAM_NAMES:
        row_str += f'  {r[f"DL_{pname}_mean"]:6.2f}  {r[f"DM_{pname}_mean"]:6.2f}'
    print(row_str)

print(f'\nCSV saved: {FIG_DIR}/e11_roi_means.csv')
df

## 7. Bar plot — ROI means across regions

In [ ]:
if len(df) == 0:
    print('No regions with sufficient voxels — check parcellation overlap.')
else:
    regions  = df['Region'].tolist()
    x        = np.arange(len(regions))
    bw       = 0.35
    fig, axes = plt.subplots(1, 4, figsize=(14, 4),
                             gridspec_kw={'wspace': 0.42})

    for ax, pname, punit, plabel in zip(axes, PARAM_NAMES, PARAM_UNITS, PARAM_LABEL):
        dl_m = df[f'DL_{pname}_mean'].values
        dl_s = df[f'DL_{pname}_std'].values
        dm_m = df[f'DM_{pname}_mean'].values
        dm_s = df[f'DM_{pname}_std'].values

        ax.bar(x - bw/2, dl_m, bw, yerr=dl_s, capsize=3,
               color=C_DL, alpha=0.8, label='Triple DL',
               error_kw=dict(lw=0.8), edgecolor='white', linewidth=0.5)
        ax.bar(x + bw/2, dm_m, bw, yerr=dm_s, capsize=3,
               color=C_DM, alpha=0.8, label='DM',
               error_kw=dict(lw=0.8), edgecolor='white', linewidth=0.5)

        ax.set_xticks(x)
        ax.set_xticklabels(regions, rotation=35, ha='right', fontsize=7.5)
        ax.set_ylabel(plabel, fontsize=9)
        ax.set_title(plabel, fontsize=10, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.yaxis.grid(True, linestyle=':', alpha=0.4)
        ax.set_axisbelow(True)
        if ax is axes[0]:
            ax.legend(fontsize=8, frameon=False)

    fig.suptitle('E11 — ROI means by parcellation region  (Air condition)',
                 fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'e11_roi_barplot.png'),
                dpi=300, bbox_inches='tight')
    plt.show()